## Task
Develop a computer vision system that, given a reference image for each book, is able to identify such book from one picture of a shelf.

<figure>
<a href="https://ibb.co/pvLVjbM5"><img src="https://i.ibb.co/svVx9bNz/example.png" alt="example" border="0"></a>
</figure>

For each type of product displayed on the shelf, the system should compute a bounding box aligned with the book spine or cover and report:
1. Number of instances;
1. Dimension of each instance (area in pixel of the bounding box that encloses each one of them);
1. Position in the image reference system of each instance (four corners of the bounding box that enclose them);
1. Overlay of the bounding boxes on the scene images.

<font color="red"><b>Each step of this assignment must be solved using traditional computer vision techniques.</b></font>

#### Example of expected output
```
Book 0 - 2 instance(s) found:
  Instance 1 {top_left: (100,200), top_right: (110, 220), bottom_left: (10, 202), bottom_right: (10, 208), area: 230px}
  Instance 2 {top_left: (90,310), top_right: (95, 340), bottom_left: (24, 205), bottom_right: (23, 234), area: 205px}
Book 1 – 1 instance(s) found:
.
.
.
```

In [3]:
import numpy as np
import cv2
from matplotlib import pyplot as plt
import os

In [4]:
# from google.colab import drive
# drive.mount('/content/drive') 

# !cp -r /content/drive/MyDrive/AssignmentsIPCV/dataset.zip ./
# !unzip dataset.zip

In [5]:
# get models and scenes images
models_path = './dataset/models/'
scenes_path = './dataset/scenes/'
imgs_model = [os.path.join(models_path, f) for f in os.listdir(models_path)]
imgs_scene = [os.path.join(scenes_path, f) for f in os.listdir(scenes_path)]

In [6]:
def locate_book_in_scene(img_model_path:str,
                         img_scene_path:str,
                         hyp_params:dict={}):
  
  # hyper-parameters:
  min_match_count = hyp_params.get('min_match_count', 50)
  good_matches_threshold = hyp_params.get('good_matches_threshold', 0.75)

  img_scene = cv2.imread(img_scene_path)
  img_model = cv2.imread(img_model_path)

  sift = cv2.SIFT_create(sigma=0.5)

  instances = []

  while True:

    kp_model = sift.detect(img_model)
    kp_scene = sift.detect(img_scene)

    kp_model, des_model = sift.compute(img_model, kp_model)
    kp_scene, des_scene = sift.compute(img_scene, kp_scene)

    FLANN_INDEX_KDTREE = 1
    index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
    search_params = dict(checks=50)
    flann = cv2.FlannBasedMatcher(index_params, search_params)

    matches = flann.knnMatch(des_model, des_scene, k=2)

    good = [m for m, n in matches if m.distance < good_matches_threshold * n.distance]

    # Corners of the model image
    h, w = img_model.shape[:2]
    pts = np.float32([[0,0], [0,h-1], [w-1,h-1], [w-1,0]]).reshape(-1, 1, 2)

    if len(good) > min_match_count:
        # Building the correspondence arrays of good matches
        src_pts = np.float32([kp_model[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
        dst_pts = np.float32([kp_scene[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
        # Using RANSAC to estimate a robust homography.
        # It returns the homography M and a mask for the discarded points.
        M, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)

        # Mask of discarded point used in visualization
        matches_mask = mask.ravel().tolist()

        # Corners of the model image
        h, w = img_model.shape[:2]
        h_t, w_t = img_scene.shape[:2]
        pts = np.float32([[0,0], [0,h-1], [w-1,h-1], [w-1,0]]).reshape(-1, 1, 2)

        # Projecting the corners into the scene image
        dst = cv2.perspectiveTransform(pts, M)

        # Saving position and area info
        instances.append({
            "dst" : dst,
            "top_left" : (int(dst[0][0][0]), int(dst[0][0][1])),
            "top_right" : (int(dst[1][0][0]), int(dst[1][0][1])),
            "bottom_left" : (int(dst[2][0][0]), int(dst[2][0][1])),
            "bottom_right" : (int(dst[3][0][0]), int(dst[3][0][1])),
        })
        # Drawing the bounding box
        img_scene = cv2.polylines(img_scene,[np.int32(dst)], True, 255, 3, cv2.LINE_AA)

        img_scene = cv2.fillPoly(img_scene, [np.int32(dst)], color=(0,0,0))

          # cv2.imwrite('./dataset/scenes/scene_10_2.jpg',img_scene)

    else:
        #print(f"Not enough matches are found - {len(good)}/{MIN_MATCH_COUNT}")
        matches_mask = None
        break

  return instances

In [7]:
models = [f'model_{i}.png' for i in range(0, 21)]
scenes = [f'scene_{i}.jpg' for i in range(0, 28)]

imgs_model = [os.path.join(models_path, f) for f in models]
imgs_scene = [os.path.join(scenes_path, f) for f in scenes]

In [8]:
import itertools

def optimize_hyperparameters(models, scenes, ground_truth, param_grid):
    best_params = None
    best_score = -1
    best_errors = None
    
    keys, values = zip(*param_grid.items())
    for combo in itertools.product(*values):
        print("Testing combo:", combo)
        params = dict(zip(keys, combo))
        all_prec, all_rec = [], []
        errors = []  # Collect mismatches for this combo

        for s_idx, scene in enumerate(scenes):
            scene_key = f"scene_{s_idx}"
            if scene_key not in ground_truth:
                continue

            gt_counts = ground_truth[scene_key]

            for m_idx, model in enumerate(models):
                gt_count = gt_counts[m_idx]
                detected = locate_book_in_scene(model, scene, params)
                pred_count = len(detected)

                # Error reporting
                if gt_count != pred_count:
                    error = {
                        'scene': scene_key,
                        'model': model,
                        'gt_count': gt_count,
                        'pred_count': pred_count,
                        'error_type': (
                            'false_positive' if gt_count == 0 and pred_count > 0 else
                            'missed_detection' if gt_count > 0 and pred_count == 0 else
                            'count_mismatch' if gt_count > 0 and pred_count > 0 else
                            'correct'  # fallback, should not happen here
                        )
                    }
                    errors.append(error)
                    print(f"  Mismatch in {scene_key} for {model}: GT={gt_count}, Pred={pred_count}")
                    

                if gt_count == 0 and pred_count == 0:
                    # both say "none" -> perfect
                    precision, recall = 1.0, 1.0
                elif gt_count == 0 and pred_count > 0:
                    # false positives
                    precision, recall = 0.0, 0.0
                elif gt_count > 0 and pred_count == 0:
                    # missed all ground truth
                    precision, recall = 0.0, 0.0
                else:
                    # normal case: both > 0
                    matched = min(pred_count, gt_count)
                    precision = matched / pred_count
                    recall = matched / gt_count

                all_prec.append(precision)
                all_rec.append(recall)

        if all_prec and all_rec:
            mean_prec = np.mean(all_prec)
            mean_rec = np.mean(all_rec)
            f1 = 2 * (mean_prec * mean_rec) / (mean_prec + mean_rec + 1e-10)

            print(f"  F1 = {f1:.3f}")
            if f1 > best_score:
                best_score = f1
                best_params = params
                best_errors = errors
    
    return best_params, best_score, best_errors

In [10]:
ground_truth = {
    # gt_from_0_to_9
    "scene_0":  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "scene_1":  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0],  # mega dubbio sul model 18, se è 1 o 2
    "scene_2":  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
    "scene_3":  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0],
    "scene_4":  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 0, 0, 0, 0, 0, 0],
    "scene_5":  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    "scene_6":  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
    "scene_7":  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0],
    "scene_8":  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "scene_9":  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 0, 0],  # dubbio sul model 19, se è 2 o 4

    # gt_from_10_to_19
    "scene_10": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0],
    "scene_11": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "scene_12": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "scene_13": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "scene_14": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "scene_15": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],  # same as 16 and 17 but with different light or color scale
    "scene_16": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],  # same as 15 and 17 but with different light or color scale
    "scene_17": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],  # same as 15 and 16 but with different light or color scale
    "scene_18": [0, 0, 0, 0, 0, 0, 0, 0, 2, 1, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],  # NOTE: model_11 instances are 2 (one ruined, considered 1); model_12 instances are 3 (differences, considered 1)
    "scene_19": [0, 0, 0, 0, 0, 0, 3, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],

    # gt
    "scene_20": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "scene_21": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "scene_22": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "scene_23": [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "scene_24": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "scene_25": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "scene_26": [2, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "scene_27": [0, 0, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],  # supersus
    "scene_28": [0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
}

param_grid = {
    "sigma": [0.4, 0.5, 0.6, 0.7],
    "min_match_count": [45, 50, 55, 60, 65],
    "good_matches_threshold": [0.6, 0.7, 0.75, 0.8, 0.85]
}

# The total number of combinations is:
total_combinations = len(param_grid["sigma"]) * len(param_grid["min_match_count"]) * len(param_grid["good_matches_threshold"])
print("Total number of combinations:", total_combinations)

best_params, best_score, best_errors = optimize_hyperparameters(imgs_model, imgs_scene, ground_truth, param_grid)

print("Best parameters:", best_params)
print("Best F1 score:", best_score)
print("The model with the best parameters did these mistakes:")

# Print a summary of errors in a readable way
if best_errors:
    print("\nMismatches (scene, model, gt_count, pred_count, error_type):")
    for err in best_errors:
        print(f"Scene: {err['scene']}, Model: {err['model']}, GT: {err['gt_count']}, Pred: {err['pred_count']}, Type: {err['error_type']}")
else:
    print("No mismatches found!")

Total number of combinations: 1
Testing combo: (0.5, 50, 0.75)
  Mismatch in scene_9 for ./dataset/models/model_19.png: GT=4, Pred=1
  Mismatch in scene_18 for ./dataset/models/model_8.png: GT=2, Pred=3
  Mismatch in scene_18 for ./dataset/models/model_9.png: GT=1, Pred=3
  Mismatch in scene_25 for ./dataset/models/model_4.png: GT=0, Pred=1
  Mismatch in scene_27 for ./dataset/models/model_2.png: GT=2, Pred=5
  Mismatch in scene_27 for ./dataset/models/model_3.png: GT=2, Pred=4
  F1 = 0.996
Best parameters: {'sigma': 0.5, 'min_match_count': 50, 'good_matches_threshold': 0.75}
Best F1 score: 0.9958745270224949
The model with the best parameters did these mistakes:

Mismatches (scene, model, gt_count, pred_count, error_type):
Scene: scene_9, Model: ./dataset/models/model_19.png, GT: 4, Pred: 1, Type: count_mismatch
Scene: scene_18, Model: ./dataset/models/model_8.png, GT: 2, Pred: 3, Type: count_mismatch
Scene: scene_18, Model: ./dataset/models/model_9.png, GT: 1, Pred: 3, Type: count_mi

In [ ]:
for i, s in enumerate(imgs_scene):
  print(f"Scene {i}")

  books = []

  for j, m in enumerate(imgs_model):

    instances = locate_book_in_scene(m, s)


    if instances:

      print(f"\tBook {j} - {len(instances)} instance(s) found:")

      books.append({
        "num" : j,
        "instances" : instances
      })

      for k, instance in enumerate(instances):

        printing_instance = {}
        printing_instance["top_left"] = instance["top_left"]
        printing_instance["top_right"] = instance["top_right"]
        printing_instance["bottom_left"] = instance["bottom_left"]
        printing_instance["bottom_right"] = instance["bottom_right"]

        print(f"\t\tInstance {k+1} {printing_instance}")

  # Plot the scene with the bounding box
  for book in books:

    img_scene = cv2.imread(s)

    for instance in book["instances"]:

      img_scene = cv2.polylines(img_scene,[np.int32(instance["dst"])], True, 255, 3, cv2.LINE_AA)

    plt.imshow(img_scene)
    plt.show()
